# 06 — Revised highlighted ingredients and claims

Prepared from the author's November 4 notebook, captured page text and manually reviewed screenshots. This publication copy preserves the selected ingredients, exact claim text, one category per claim, standardization mapping and output table schemas. Local originals are unchanged.

## Source and coding choices

- BOJ cleanser: parse the captured November page text. The original live request is optional; it is disabled to avoid changing the snapshot or failing on HTTP 429.
- Other products: manually transcribed screenshots. Ingredient descriptions not visible remain missing. A description shared by a combined ingredient heading is retained for each named ingredient.
- COSRX cleanser/toner: Benefits. COSRX cream/serum: Key Benefit column. LANEIGE: visible Benefits icon labels; its closed expanded panels are not inferred. Cream Skin's two ingredient labels/descriptions come from the expanded Benefits section.
- Each claim receives one category: heading's main message, description to clarify ambiguity, first explicit benefit for equally weighted compound statements. Original descriptions survive. Usage and testing labels remain separate claim types.
- Percentages are retained separately from comparison names. The snail-label mapping is for marketing analysis, not proof of chemical identity. Other peptide/complex labels remain distinct.
- Original processing timestamps are preserved below; they do not establish when older reviews were collected. New run/export time is recorded separately.
- Bundle, special-edition and historical formula differences remain unresolved source limitations. Highlighted ingredients are not full-formula presence or efficacy evidence.

## Run

Set the project root in the next cell. For product FK validation, supply local `data/tableau/products.csv` (from 05) or set `KBEAUTY_PRODUCTS_CSV`. No raw review data is needed for this notebook. Outputs go into a new timestamped folder, never the original input files.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
import os
import pandas as pd
from IPython.display import display
from datetime import datetime, timezone

PRODUCTS_PATH = Path(os.environ.get("KBEAUTY_PRODUCTS_CSV", str(PROJECT_ROOT / "data/tableau/products.csv")))
if not PRODUCTS_PATH.is_file():
    raise FileNotFoundError("Supply local products.csv or set KBEAUTY_PRODUCTS_CSV before running.")
products_for_validation = pd.read_csv(PRODUCTS_PATH, dtype="string", keep_default_na=False)
if "product_id" not in products_for_validation or products_for_validation.product_id.eq("").any() or not products_for_validation.product_id.is_unique:
    raise ValueError("Products requires unique, nonblank product_id.")
SNAPSHOT_PROCESSED_AT = {'BOJ_CLEANSER_01': '2026-11-04T14:17:01.795310+00:00', 'BOJ_MOISTURIZER_01': '2026-11-04T14:25:37.756194+00:00', 'BOJ_SERUM_01': '2026-11-04T14:29:19.695452+00:00', 'BOJ_TONER_01': '2026-11-04T14:31:18.014541+00:00', 'COSRX_CLEANSER_01': '2026-11-04T14:49:08.751933+00:00', 'COSRX_MOISTURIZER_01': '2026-11-04T14:49:08.751933+00:00', 'COSRX_SERUM_01': '2026-11-04T14:49:08.751933+00:00', 'COSRX_TONER_01': '2026-11-04T14:49:08.751933+00:00', 'LANEIGE_CLEANSER_01': '2026-11-04T14:49:08.751933+00:00', 'LANEIGE_MOISTURIZER_01': '2026-11-04T14:49:08.751933+00:00', 'LANEIGE_SERUM_01': '2026-11-04T14:49:08.751933+00:00', 'LANEIGE_TONER_01': '2026-11-04T14:49:08.751933+00:00'}
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()


## 1. Cleanser captured text (offline by default)

In [ ]:
REFRESH_CLEANSER_PAGE = False
page_text = 'Comfortably Cleanses\nRefresh your skin with a pH-friendly formula.\nSupports Moisture Barrier\nAdd mung bean extract to support barrier moisture.\nPreps for Skincare\nLeave your face moisturized, soft, and ready for skincare with vitamin-rich plum water.\nKey Ingredients\nPlum Water\nThis vitamin-rich ingredient helps hydrate. Contains natural fruit acids that help soften dead skin cells.\nMung Bean Extract\nContains natural saponins that add a boost of moisture.\nFull Ingredient List'
product_id = "BOJ_CLEANSER_01"
product_url = "https://beautyofjoseon.com/products/green-plum-refreshing-cleanser"
if REFRESH_CLEANSER_PAGE:
    import requests
    from bs4 import BeautifulSoup
    response = requests.get(product_url, timeout=30)
    response.raise_for_status()  # Stop on 429; do not repeatedly retry.
    page_text = BeautifulSoup(response.text, "html.parser").get_text("\n", strip=True)
    SNAPSHOT_PROCESSED_AT[product_id] = datetime.now(timezone.utc).isoformat()


## 2. Extract cleanser and assign categories

In [ ]:
import re
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

# Normalize whitespace to make section matching reliable.
text = re.sub(r"\s+", " ", page_text).strip()

def extract_between(text, start, end):
    start_position = text.find(start)
    if start_position == -1:
        raise ValueError(f"Missing section: {start}")

    start_position += len(start)
    end_position = text.find(end, start_position)
    if end_position == -1:
        raise ValueError(f"Missing section boundary: {end}")

    result = text[start_position:end_position].strip()
    if not result:
        raise ValueError(f"Empty content after: {start}")
    return result

# One category per claim, based on its main heading.
claim_definitions = [
    ("Comfortably Cleanses", "Supports Moisture Barrier", "Cleansing"),
    ("Supports Moisture Barrier", "Preps for Skincare", "Barrier support"),
    ("Preps for Skincare", "Key Ingredients", "Skincare preparation"),
]

# Limit ingredient extraction to the highlighted-ingredients section.
ingredient_section = extract_between(
    text, "Key Ingredients", "Full Ingredient List"
)

ingredient_definitions = [
    ("Plum Water", "Mung Bean Extract"),
    ("Mung Bean Extract", None),
]

# Time this downloaded page was processed; not the historical collection date.
processed_at = SNAPSHOT_PROCESSED_AT["BOJ_CLEANSER_01"]

claim_rows = [
    {
        "product_id": product_id,
        "claim_text": heading,
        "claim_description": extract_between(text, heading, next_heading),
        "claim_category": category,
        "source_url": product_url,
        "processed_at_utc": processed_at,
        "coding_method": "Manual category assignment based on claim heading",
    }
    for heading, next_heading, category in claim_definitions
]

ingredient_rows = []

for name, next_name in ingredient_definitions:
    if next_name:
        description = extract_between(
            ingredient_section, name, next_name
        )
    else:
        position = ingredient_section.find(name)
        if position == -1:
            raise ValueError(f"Missing key ingredient: {name}")
        description = ingredient_section[position + len(name):].strip()

    ingredient_rows.append({
        "product_id": product_id,
        "key_ingredient_name": name,
        "ingredient_description": description,
        "source_url": product_url,
        "processed_at_utc": processed_at,
    })

boj_cleanser_claims = pd.DataFrame(claim_rows)
boj_cleanser_key_ingredients = pd.DataFrame(ingredient_rows)

print("KEY INGREDIENTS")
display(boj_cleanser_key_ingredients)

print("CLAIMS — ONE CATEGORY PER CLAIM")
display(boj_cleanser_claims)

In [ ]:
# Store reviewed products separately as we work through the sample.
# Running this again replaces this product's entry without creating duplicates.
if "key_ingredients_by_product" not in globals():
    key_ingredients_by_product = {}

if "claims_by_product" not in globals():
    claims_by_product = {}

key_ingredients_by_product["BOJ_CLEANSER_01"] = (
    boj_cleanser_key_ingredients.copy()
)

claims_by_product["BOJ_CLEANSER_01"] = (
    boj_cleanser_claims.copy()
)

print("Green Plum Refreshing Cleanser stored.")

## 3. Manually reviewed Beauty of Joseon positioning

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

dynasty_product_id = "BOJ_MOISTURIZER_01"
dynasty_url = "https://beautyofjoseon.com/products/dynasty-cream"
processed_at = SNAPSHOT_PROCESSED_AT["BOJ_MOISTURIZER_01"]

dynasty_ingredients = [
    "Rice Bran Water",
    "Ginseng Root Water",
    "Squalane",
    "Niacinamide",
]

boj_dynasty_key_ingredients = pd.DataFrame([
    {
        "product_id": dynasty_product_id,
        "key_ingredient_name": name,
        "ingredient_description": pd.NA,
        "source_url": dynasty_url,
        "processed_at_utc": processed_at,
        "extraction_method": "Manual transcription from supplied screenshot",
        "description_status": "Not visible in screenshot",
    }
    for name in dynasty_ingredients
])

dynasty_claims = [
    (
        "Deeply Hydrates",
        "Deliver deep hydration with rice bran water and ginseng root water.",
        "Hydration",
    ),
    (
        "Keeps Skin Balanced",
        "Keep oil-moisture levels balanced and address the appearance "
        "of skin texture with niacinamide and squalane.",
        "Oil–moisture balance",
    ),
    (
        "Gives a Dewy Glow",
        "Get the dewy glass-skin look with this non-comedogenic "
        "tested, makeup-friendly moisturizer.",
        "Radiance / glow",
    ),
]

boj_dynasty_claims = pd.DataFrame([
    {
        "product_id": dynasty_product_id,
        "claim_text": heading,
        "claim_description": description,
        "claim_category": category,
        "source_url": dynasty_url,
        "processed_at_utc": processed_at,
        "coding_method": "Manual category assignment based on claim heading",
        "extraction_method": "Manual transcription from supplied screenshot",
    }
    for heading, description, category in dynasty_claims
])

with pd.option_context("display.max_colwidth", None):
    print("DYNASTY CREAM — KEY INGREDIENTS")
    display(boj_dynasty_key_ingredients)

    print("DYNASTY CREAM — CLAIMS")
    display(boj_dynasty_claims)

In [ ]:
if "key_ingredients_by_product" not in globals():
    key_ingredients_by_product = {}

if "claims_by_product" not in globals():
    claims_by_product = {}

key_ingredients_by_product["BOJ_MOISTURIZER_01"] = (
    boj_dynasty_key_ingredients.copy()
)

claims_by_product["BOJ_MOISTURIZER_01"] = (
    boj_dynasty_claims.copy()
)

print("Products stored:", list(claims_by_product))

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

glow_product_id = "BOJ_SERUM_01"
glow_url = (
    "https://beautyofjoseon.com/products/"
    "glow-serum-propolis-niacinamide"
)
glow_processed_at = SNAPSHOT_PROCESSED_AT["BOJ_SERUM_01"]

glow_ingredient_names = [
    "Niacinamide",
    "Propolis Extract",
    "BHA (Beta Hydroxy Acid)",
]

boj_glow_key_ingredients = pd.DataFrame([
    {
        "product_id": glow_product_id,
        "key_ingredient_name": name,
        "ingredient_description": pd.NA,
        "source_url": glow_url,
        "processed_at_utc": glow_processed_at,
        "extraction_method": "Manual transcription from supplied screenshot",
        "description_status": "Not visible in screenshot",
    }
    for name in glow_ingredient_names
])

glow_claim_definitions = [
    (
        "Boosts Glow with BHA",
        "Exfoliate and clear pores with BHA for smoother-looking, "
        "brighter-looking glass skin.",
        "Radiance / glow",
    ),
    (
        "Calms + Soothes",
        "Help hydrate and calm skin with propolis extract.",
        "Soothing",
    ),
    (
        "Balances Sebum",
        "Keep skin's oil-moisture levels in check with niacinamide.",
        "Oil–moisture balance",
    ),
]

boj_glow_claims = pd.DataFrame([
    {
        "product_id": glow_product_id,
        "claim_text": heading,
        "claim_description": description,
        "claim_category": category,
        "source_url": glow_url,
        "processed_at_utc": glow_processed_at,
        "coding_method": "Manual category assignment based on claim heading",
        "extraction_method": "Manual transcription from supplied screenshot",
    }
    for heading, description, category in glow_claim_definitions
])

with pd.option_context("display.max_colwidth", None):
    print("GLOW SERUM — KEY INGREDIENTS")
    display(boj_glow_key_ingredients)

    print("GLOW SERUM — CLAIMS")
    display(boj_glow_claims)

In [ ]:
# Run the earlier storage cells first if these dictionaries are missing.
key_ingredients_by_product["BOJ_SERUM_01"] = (
    boj_glow_key_ingredients.copy()
)

claims_by_product["BOJ_SERUM_01"] = (
    boj_glow_claims.copy()
)

print("Products stored:", list(claims_by_product))

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

ginseng_product_id = "BOJ_TONER_01"
ginseng_url = "https://beautyofjoseon.com/products/ginseng-essence-water"
ginseng_processed_at = SNAPSHOT_PROCESSED_AT["BOJ_TONER_01"]

ginseng_ingredient_names = [
    "Ginseng Root Water",
    "Ginseng Complex",
    "Niacinamide",
]

boj_ginseng_key_ingredients = pd.DataFrame([
    {
        "product_id": ginseng_product_id,
        "key_ingredient_name": name,
        "ingredient_description": pd.NA,
        "source_url": ginseng_url,
        "processed_at_utc": ginseng_processed_at,
        "extraction_method": "Manual transcription from supplied screenshot",
        "description_status": "Not visible in screenshot",
    }
    for name in ginseng_ingredient_names
])

ginseng_claim_definitions = [
    (
        "Floods Skin With Ginseng",
        "Nourish your skin with ginseng root water + ginseng "
        "complex for lasting moisture.",
        "Hydration",
        "Category clarified by supporting description",
    ),
    (
        "Supports Moisture Barrier",
        "Start your routine with this comfortably lightweight, "
        "fast-absorbing layer that helps support the moisture barrier.",
        "Barrier support",
        "Category based on claim heading",
    ),
    (
        "Maintains Oil-Moisture Balance",
        "Keep your skin's oil-moisture levels balanced with niacinamide.",
        "Oil–moisture balance",
        "Category based on claim heading",
    ),
]

boj_ginseng_claims = pd.DataFrame([
    {
        "product_id": ginseng_product_id,
        "claim_text": heading,
        "claim_description": description,
        "claim_category": category,
        "source_url": ginseng_url,
        "processed_at_utc": ginseng_processed_at,
        "coding_method": "Manual category assignment",
        "coding_note": note,
        "extraction_method": "Manual transcription from supplied screenshot",
    }
    for heading, description, category, note in ginseng_claim_definitions
])

with pd.option_context("display.max_colwidth", None):
    print("GINSENG ESSENCE WATER — KEY INGREDIENTS")
    display(boj_ginseng_key_ingredients)

    print("GINSENG ESSENCE WATER — CLAIMS")
    display(boj_ginseng_claims)

In [ ]:
key_ingredients_by_product["BOJ_TONER_01"] = (
    boj_ginseng_key_ingredients.copy()
)

claims_by_product["BOJ_TONER_01"] = (
    boj_ginseng_claims.copy()
)

boj_ids = [
    "BOJ_CLEANSER_01",
    "BOJ_MOISTURIZER_01",
    "BOJ_SERUM_01",
    "BOJ_TONER_01",
]

for product_id in boj_ids:
    if (
        product_id not in key_ingredients_by_product
        or product_id not in claims_by_product
    ):
        raise ValueError(f"Missing stored tables: {product_id}")

boj_key_ingredients = pd.concat(
    [key_ingredients_by_product[p] for p in boj_ids],
    ignore_index=True,
)

boj_claims = pd.concat(
    [claims_by_product[p] for p in boj_ids],
    ignore_index=True,
)

print("Products:", boj_claims["product_id"].nunique())
print("Key ingredient entries:", len(boj_key_ingredients))
print("Claims:", len(boj_claims))

display(
    boj_claims[
        ["product_id", "claim_text", "claim_category"]
    ]
)

## 4. Manually reviewed COSRX and LANEIGE positioning

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

processed_at = SNAPSHOT_PROCESSED_AT["COSRX_CLEANSER_01"]

# Each claim tuple contains:
# (exact claim text, one category, claim type)
#
# Ingredient labels are preserved as displayed.
# Different labels such as "Snail Mucin 92%" and "Snail Filtrate"
# remain separate until we explicitly review ingredient standardization.

remaining_products = {
    "COSRX_CLEANSER_01": {
        "url": (
            "https://www.cosrx.com/collections/cleansers/products/"
            "low-ph-good-morning-gel-cleanser?variant=37517677953180"
        ),
        "ingredient_section": "Ingredients > Key Ingredients",
        "claim_section": "Details > Benefits",
        "ingredients": [
            (
                "Betaine Salicylate (BHA)",
                "Soothe and strengthen for luminous and clear skin",
            ),
            (
                "Tea Tree Oil",
                "Soothe and strengthen for luminous and clear skin",
            ),
        ],
        "claims": [
            ("Hydrating cleanse without stripping", "Cleansing", "Skin benefit"),
            ("Removes impurities", "Cleansing", "Skin benefit"),
            ("Protects skin barrier", "Barrier support", "Skin benefit"),
            ("Restores pH Balance", "pH positioning", "Skin benefit"),
        ],
    },

    "COSRX_MOISTURIZER_01": {
        "url": (
            "https://www.cosrx.com/collections/moisturizer-cream/products/"
            "advanced-snail-92-all-in-one-cream"
        ),
        "ingredient_section": "Concerns & Benefits > Key Ingredient",
        "claim_section": "Concerns & Benefits > Key Benefit",
        "ingredients": [
            ("Snail Mucin 92%", None),
            ("Sodium Hyaluronate", None),
            ("Snail Filtrate", None),
            ("Betaine", None),
        ],
        "claims": [
            ("Soothes and calms skin", "Soothing", "Skin benefit"),
            ("Deep hydration, fresh glow", "Hydration", "Skin benefit"),
            (
                "Smooths and softens",
                "Skin texture / smoothness",
                "Skin benefit",
            ),
            (
                "Replenishes moisture barrier",
                "Barrier support",
                "Skin benefit",
            ),
        ],
    },

    "COSRX_SERUM_01": {
        "url": (
            "https://www.cosrx.com/collections/serums-essences/products/"
            "the-6-peptide-skin-booster-serum-bundle"
        ),
        "ingredient_section": "Concerns & Benefits > Key Ingredient",
        "claim_section": "Concerns & Benefits > Key Benefit",
        "ingredients": [
            ("6 Peptide Complex", None),
            ("Niacinamide", None),
            ("Hyaluronic Acid", None),
            ("Peptides", None),
            ("Peptide Blend", None),
            ("Adenosine", None),
        ],
        "claims": [
            (
                "Firms and improves elasticity",
                "Firmness / elasticity",
                "Skin benefit",
            ),
            ("Boosts glow & vitality", "Radiance / glow", "Skin benefit"),
            ("Brightens and refines skin", "Radiance / glow", "Skin benefit"),
            (
                "Visibly lifts and tightens",
                "Firmness / elasticity",
                "Skin benefit",
            ),
            (
                "Smooths and refines surface",
                "Skin texture / smoothness",
                "Skin benefit",
            ),
        ],
    },

    "COSRX_TONER_01": {
        "url": (
            "https://www.cosrx.com/collections/toners/products/"
            "full-fit-propolis-synergy-toner?variant=43487462129880"
        ),
        "ingredient_section": "Ingredients > Key Ingredients",
        "claim_section": "Details > Benefits",
        "ingredients": [
            (
                "72.6% Propolis Extract",
                "Nourish tired skin leaving it healthy and plump",
            ),
            (
                "10.7% Honey Extract",
                "Charge the skin with revitalizing energy",
            ),
            ("0.2% Panthenol", None),
        ],
        "claims": [
            ("Nourish for healthy skin", "Nourishment", "Skin benefit"),
            ("Plump, glowing skin", "Plumping", "Skin benefit"),
            ("Reduce redness", "Redness reduction", "Skin benefit"),
            (
                "Improve skin texture",
                "Skin texture / smoothness",
                "Skin benefit",
            ),
        ],
    },

    "LANEIGE_CLEANSER_01": {
        "url": (
            "https://us.laneige.com/products/"
            "water-bank-gentle-gel-cleanser?variant=45866099671092"
        ),
        "ingredient_section": "Summary > Key Ingredients",
        "claim_section": "Summary > Benefits icons",
        "ingredients": [
            (
                "Blue Hyaluronic Acid (Blue HA)",
                "Delivers effective, long-lasting hydration",
            ),
            ("Allantoin", "Soothes skin."),
            ("Amino-acid Complex", "Supports skin's barrier."),
        ],
        "claims": [
            ("AM & PM", "Usage timing", "Usage"),
            ("Derm Tested", "Dermatologist testing", "Testing statement"),
            ("Hydration", "Hydration", "Skin benefit"),
        ],
    },

    "LANEIGE_MOISTURIZER_01": {
        "url": (
            "https://us.laneige.com/products/"
            "water-bank-blue-hyaluronic-cream-moisturizer"
            "?variant=48218677739572"
        ),
        "ingredient_section": "Summary > Key Ingredient",
        "claim_section": "Summary > Benefits icons",
        "ingredients": [
            ("Blue Hyaluronic Acid", None),
            ("Ceramide", None),
            ("Squalane", None),
        ],
        "claims": [
            ("AM & PM", "Usage timing", "Usage"),
            ("Derm Tested", "Dermatologist testing", "Testing statement"),
            ("Hydration", "Hydration", "Skin benefit"),
            ("Soften", "Skin texture / smoothness", "Skin benefit"),
        ],
    },

    "LANEIGE_SERUM_01": {
        "url": (
            "https://us.laneige.com/products/"
            "water-bank-blue-hyaluronic-serum?variant=39875620864052"
        ),
        "ingredient_section": "Summary > Key Ingredient",
        "claim_section": "Summary > Benefits icons",
        "ingredients": [
            ("Blue Hyaluronic Acid", None),
            ("Peptide", None),
        ],
        "claims": [
            ("AM & PM", "Usage timing", "Usage"),
            ("Derm Tested", "Dermatologist testing", "Testing statement"),
            ("Hydration", "Hydration", "Skin benefit"),
        ],
    },

    "LANEIGE_TONER_01": {
        "url": (
            "https://us.laneige.com/products/"
            "cream-skin-toner-moisturizer?variant=41084692267060"
        ),
        "ingredient_section": "Benefits > Highlighted ingredients",
        "claim_section": "Summary > Benefits icons",
        "ingredients": [
            (
                "Ceramide and Peptide Complex",
                "Provides intense, nourishing hydration and visibly "
                "firming benefits",
            ),
            (
                "White Leaf Tea",
                "Rich in moisturizing amino acids to help soothe skin",
            ),
        ],
        "claims": [
            ("AM & PM", "Usage timing", "Usage"),
            ("Derm Tested", "Dermatologist testing", "Testing statement"),
            ("Hydration", "Hydration", "Skin benefit"),
            ("Soften", "Skin texture / smoothness", "Skin benefit"),
            ("Soothe", "Soothing", "Skin benefit"),
        ],
    },
}

remaining_ingredient_rows = []
remaining_claim_rows = []

for product_id, product in remaining_products.items():
    for name, description in product["ingredients"]:
        remaining_ingredient_rows.append({
            "product_id": product_id,
            "key_ingredient_name": name,
            "ingredient_description": (
                description if description is not None else pd.NA
            ),
            "source_url": product["url"],
            "source_section": product["ingredient_section"],
            "processed_at_utc": SNAPSHOT_PROCESSED_AT[product_id],
            "extraction_method": (
                "Manual transcription from supplied screenshot"
            ),
            "description_status": (
                "Visible in screenshot"
                if description is not None
                else "No separate description shown in screenshot"
            ),
        })

    for text, category, claim_type in product["claims"]:
        remaining_claim_rows.append({
            "product_id": product_id,
            "claim_text": text,
            # These sections show standalone bullets or labels,
            # rather than headings with separate descriptions.
            "claim_description": pd.NA,
            "claim_category": category,
            "claim_type": claim_type,
            "source_url": product["url"],
            "source_section": product["claim_section"],
            "processed_at_utc": SNAPSHOT_PROCESSED_AT[product_id],
            "coding_method": "Manual category assignment",
            "coding_note": (
                "One category per claim; main benefit selected. "
                "For equally explicit combined benefits, first benefit used."
            ),
            "extraction_method": (
                "Manual transcription from supplied screenshot"
            ),
        })

remaining_key_ingredients = pd.DataFrame(remaining_ingredient_rows)
remaining_claims = pd.DataFrame(remaining_claim_rows)

# Validate before adding anything to the reviewed dictionaries.
if remaining_key_ingredients.duplicated(
    ["product_id", "key_ingredient_name"]
).any():
    raise ValueError("Duplicate ingredient label within a product.")

if remaining_claims.duplicated(
    ["product_id", "claim_text"]
).any():
    raise ValueError("More than one row/category for a claim.")

if remaining_claims["claim_category"].isna().any():
    raise ValueError("A claim is missing its category.")

summary = (
    remaining_key_ingredients.groupby("product_id")
    .size()
    .rename("key_ingredient_entries")
    .to_frame()
    .join(
        remaining_claims.groupby("product_id")
        .size()
        .rename("claims")
    )
)

display(summary)

with pd.option_context("display.max_colwidth", None):
    display(
        remaining_key_ingredients[
            ["product_id", "key_ingredient_name", "ingredient_description"]
        ]
    )
    display(
        remaining_claims[
            ["product_id", "claim_text", "claim_category", "claim_type"]
        ]
    )

## 5. Combine and validate claim types

In [ ]:
# Add the eight products to the reviewed collections.
for product_id in remaining_products:
    key_ingredients_by_product[product_id] = (
        remaining_key_ingredients.loc[
            remaining_key_ingredients["product_id"].eq(product_id)
        ].copy()
    )

    claims_by_product[product_id] = (
        remaining_claims.loc[
            remaining_claims["product_id"].eq(product_id)
        ].copy()
    )

all_key_ingredients = pd.concat(
    key_ingredients_by_product.values(),
    ignore_index=True,
)

all_product_claims = pd.concat(
    claims_by_product.values(),
    ignore_index=True,
)

# Beauty of Joseon's collected claims are all skin-benefit statements.
boj_mask = all_product_claims["product_id"].str.startswith("BOJ_")
all_product_claims.loc[boj_mask, "claim_type"] = "Skin benefit"

# Confirm one category per original product claim.
if all_product_claims.duplicated(["product_id", "claim_text"]).any():
    raise ValueError("Duplicate claim rows: review before continuing.")

if all_product_claims["claim_category"].isna().any():
    raise ValueError("Missing claim category.")

if all_product_claims["claim_type"].isna().any():
    raise ValueError("Missing claim type.")

print("Products:", all_product_claims["product_id"].nunique())
print("Key ingredient entries:", len(all_key_ingredients))
print("All claim statements:", len(all_product_claims))

print("\nCLAIM TYPES")
display(all_product_claims["claim_type"].value_counts())

## 6. Standardize ingredient comparison names

In [ ]:
# Explicit mappings keep every standardization decision visible.
ingredient_name_mapping = {
    "72.6% Propolis Extract": "Propolis Extract",
    "10.7% Honey Extract": "Honey Extract",
    "0.2% Panthenol": "Panthenol",
    "Blue Hyaluronic Acid (Blue HA)": "Blue Hyaluronic Acid",
    "Snail Mucin 92%": "Snail mucin",
    "Snail Filtrate": "Snail mucin",
}

all_key_ingredients["key_ingredient_standardized"] = (
    all_key_ingredients["key_ingredient_name"]
    .replace(ingredient_name_mapping)
)

# Preserve percentages as brand-stated information, separate from names.
all_key_ingredients["brand_stated_percentage"] = pd.to_numeric(
    all_key_ingredients["key_ingredient_name"]
    .str.extract(r"(\d+(?:\.\d+)?)%", expand=False),
    errors="coerce",
)

# One row per product/standardized ingredient for Tableau counting.
# The evidence table above retains all 37 original entries.
product_key_ingredients = (
    all_key_ingredients[
        ["product_id", "key_ingredient_standardized"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Original ingredient entries:", len(all_key_ingredients))
print("Distinct product–ingredient pairs:", len(product_key_ingredients))

display(
    all_key_ingredients[
        [
            "product_id",
            "key_ingredient_name",
            "key_ingredient_standardized",
            "brand_stated_percentage",
        ]
    ]
)

## 7. Distinct product–category membership

In [ ]:
# Preserve all original statements in all_product_claims.
# Create a separate membership table for Tableau comparisons.
product_claim_categories = (
    all_product_claims[
        ["product_id", "claim_category", "claim_type"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Original claim statements:", len(all_product_claims))
print("Distinct product–category pairs:", len(product_claim_categories))

display(
    product_claim_categories.groupby(
        ["claim_type", "claim_category"]
    )["product_id"]
    .nunique()
    .rename("products")
    .reset_index()
    .sort_values(
        ["claim_type", "products"],
        ascending=[True, False],
    )
)

## 8. Validate keys, row preservation and export

Validation checks the historical offline snapshot counts before publishing a new run folder. A live refresh must be reviewed against a new version, not silently forced into this snapshot.

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

for name, table, key in [
    ("ingredient evidence", all_key_ingredients, ["product_id", "key_ingredient_name"]),
    ("claim evidence", all_product_claims, ["product_id", "claim_text"]),
    ("ingredient pairs", product_key_ingredients, ["product_id", "key_ingredient_standardized"]),
    ("claim categories", product_claim_categories, ["product_id", "claim_category", "claim_type"]),
]:
    require(table[key].notna().all().all(), f"Null key: {name}")
    require(table[key].astype("string").apply(lambda s: s.str.strip().ne("")).all().all(), f"Blank key: {name}")
    require(not table.duplicated(key).any(), f"Duplicate key: {name}")
    require(table.product_id.isin(products_for_validation.product_id).all(), f"Unknown product: {name}")
    require(set(table.product_id) == set(products_for_validation.product_id), f"Product coverage: {name}")

require(len(all_key_ingredients) == 37 and len(product_key_ingredients) == 36, "Ingredient snapshot count mismatch")
require(len(all_product_claims) == 44 and len(product_claim_categories) == 41, "Claim snapshot count mismatch")
require(all_product_claims.claim_type.value_counts().to_dict() == {"Skin benefit": 36, "Usage": 4, "Testing statement": 4}, "Unexpected claim types")
pd.testing.assert_frame_equal(
    product_key_ingredients.reset_index(drop=True),
    all_key_ingredients[["product_id", "key_ingredient_standardized"]].drop_duplicates().reset_index(drop=True),
)
pd.testing.assert_frame_equal(
    product_claim_categories.reset_index(drop=True),
    all_product_claims[["product_id", "claim_category", "claim_type"]].drop_duplicates().reset_index(drop=True),
)
print("Snapshot keys, product coverage and membership reconstruction passed.")


In [ ]:
from pathlib import Path
from datetime import datetime

# Save in a new folder under the current notebook directory.
export_folder = (
    PROJECT_ROOT
    / "data"
    / "tableau_key_ingredients_claims"
    / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
export_folder.mkdir(parents=True, exist_ok=False)

tables_to_export = {
    "key_ingredient_evidence": all_key_ingredients,
    "claim_evidence": all_product_claims,
    "product_key_ingredients": product_key_ingredients,
    "product_claim_categories": product_claim_categories,
}

for name, table in tables_to_export.items():
    path = export_folder / f"{name}.csv"
    table.to_csv(path, index=False)

    # Confirm that exported values and columns can be read back.
    saved = pd.read_csv(path, dtype="string", keep_default_na=False)
    expected = table.astype("string").fillna("").reset_index(drop=True)
    pd.testing.assert_frame_equal(saved, expected, check_dtype=False)

    print(f"{name}.csv: {len(table)} rows")

print("\nSaved to:", export_folder.resolve())
import json, hashlib
manifest = {
    "run_started_at_utc": RUN_STARTED_AT_UTC,
    "products_input_sha256": hashlib.sha256(PRODUCTS_PATH.read_bytes()).hexdigest(),
    "live_cleanser_refresh": REFRESH_CLEANSER_PAGE,
    "original_processing_timestamps": SNAPSHOT_PROCESSED_AT,
    "outputs": {name: {"rows": len(table), "sha256": hashlib.sha256((export_folder / f"{name}.csv").read_bytes()).hexdigest()}
                for name, table in tables_to_export.items()},
}
(export_folder / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
